# 📌 Key-Value Cache

![Topic](https://img.shields.io/badge/Key-Value_Cache-blue?style=flat-square)
![Category](https://img.shields.io/badge/Category-Architecture-blueviolet?style=flat-square)
![Level](https://img.shields.io/badge/Level-Intermediate-yellow?style=flat-square)
![Last Updated](https://img.shields.io/badge/Updated-March%202026-blue?style=flat-square)
<br>
<br>
<br>
> <span style="font-size:20px;">**TL;DR** — The KV Cache is a memory optimization trick. During text generation, instead of recalculating every previous word every single time a new word is added, the model "saves" the intermediate results (the Keys and Values) in memory.</span>

---
## 1. Overview

<!-- What is this concept? 2–4 sentences that a newcomer could understand.
     Include: what problem it solves, why it exists, where it fits in the AI landscape. -->

In clasic Transformers system, every time a token is generated, the LLM must backtrack all previous tokens in the sequence.

For example, if you generate a 100-word story, the model would normally have to reprocess the first word around a hundred times. This system results in a waste of resources due to these additional calculations.

To solve this problem, the **KV cache** stores the calculations from previous steps to avoid recalculation, improving latency and efficiency.

---
## 2. How It Works

<!-- Break the concept into numbered steps or subsections.
     Use diagrams (images from assets/) where helpful.
     Each subsection should follow: description → danger/difficulty level → counter-technique or note -->

To understand the cache, you have to understand the **three pillars of Attention**:

* **Query (Q)**: Represents what the current token is **asking about**.
* **Key (K)**: Represents what each token **advertises about itself**.
* **Value (V)**: Represents the **actual content that gets aggregated**.

When the model generates the next token:

1. It computes the Q,K, and V for the **new token**.

2. It fetches the K and V **for all previous tokens** from the Cache.

3. It performs **attention math** using the new Q and the combined Keys and Values.

4. It **adds** the new K and V to the Cache for the next step.

![KV_Cache](../assets/KV_Cache.png)


---
## 3. Advantages & Limitations

| | Aspect | Commentary |
|--|--------|------------|
| 🟢 | **Speed (Latency)** | Without it, "Classic" LLM generation slows down linearly as the text gets longer. |
| 🟢 | **Efficiency** | Drastically reduces the math operations required per token. |
| 🔴 | **Memory Wall** | The cache lives in GPU VRAM. As the context grows, the cache grows.|
| 🔴 | **Throughput** | More cache means less room for Batching (processing multiple users at once) |

---
## 4. Code Example

> **Goal:** This code shows a simple simulation of a Key-Value Cache. No external dependencies are needed.

In [ ]:
# ---------------------------------------------------------------------------
# Simple simulation of a Key-Value Cache
# ---------------------------------------------------------------------------

class SimpleKVCacheLLM:
    def __init__(self):
        # Our "Memory" - stores K and V pairs for every token
        self.kv_cache = [] 

    def generate_token(self, new_token_id):
        print(f"--- Processing Token: {new_token_id} ---")
        
        # 1. Compute Q, K, V for the CURRENT token only
        # (In reality, these are high-dimensional vectors)
        current_k = f"Key_{new_token_id}"
        current_v = f"Value_{new_token_id}"
        current_q = f"Query_{new_token_id}"

        # 2. Retrieve past context from cache
        past_keys = [item['k'] for item in self.kv_cache]
        past_values = [item['v'] for item in self.kv_cache]

        # 3. Perform Attention (Conceptual)
        # We look at current_q vs (past_keys + current_k)
        all_keys = past_keys + [current_k]
        print(f"Attention math performed using {len(all_keys)} keys.")

        # 4. UPDATE CACHE: Save current K, V for the next word
        self.kv_cache.append({'k': current_k, 'v': current_v})
        
        return f"Word_{new_token_id+1}"

# Simulation
llm = SimpleKVCacheLLM()
next_word = 101

for _ in range(3):
    next_word_id = llm.generate_token(next_word)
    next_word += 1

---
## 5. Key Takeaways
<div style="font-size: 16px; line-height: 1.6;">

- **KV Cache** prevents the model from re-calculating the same math for the same words over and over.
- Although the model is **faster**, KV cache uses significant GPU memory 
- **KV Cache directly influences Batching**, forcing a choice between a long context window for one person or short context windows for many people at once.
- To **counter limitation** of KV Cache, new techniques have emerged like Multi-Query Attention (MQA) or PagedAttention.


</div>